# 03. Agent Skills — 필요할 때만 읽는 절차

## 학습 목표

- 프롬프트, 도구, Skill의 차이를 설명한다.
- 다섯 줄짜리 `SKILL.md`를 손으로 쓰고, Agent가 그것을 읽는지 확인한다.
- `description`을 바꿔 활성화가 달라지는지, 경쟁하는 Skill 셋에서 여러 번 돌려 비교한다.
- 출력 형식을 더하고, 참조 파일로 나누고, 읽으면 안 되는 질문(near-miss)을 시험한다.
- 내 업무 Skill을 하나 만든다.

## 호출 규모

이 노트북은 **실제 모델**만 씁니다. `Day02_new/.env`의 `ANTHROPIC_API_KEY`가 없으면 첫 셀에서 안내와 함께 멈춥니다. 유료 셀은 실행 직전에 규모를 한 줄로 알려 줍니다. 기본값 `RUN_OPTIONAL = False`에서 모델 호출은 저자 실행에서 약 80~90회였습니다(Agent가 도구를 몇 번 쓰느냐에 따라 달라집니다). 큰 측정(질문 20개 x 3회, 저자의 1회 실행에서 81회)은 `RUN_OPTIONAL = True`로 바꿔야만 돕니다. 비용은 본인 키로 청구됩니다.

| 구분 | 셀 수 | 모델 호출 |
|---|---:|---|
| 무료 (파일 쓰기, 검사) | 6 | 0 |
| 짧은 관찰 (Agent 한 번 또는 두 번) | 9 | 약 3 또는 6 |
| 경쟁 Skill 셋 비교 (질문 4개 x 설명 2가지 x 3회) | 1 | 약 70 |
| 트리거 테스트 네 문항 | 1 | 약 12 |
| 제공 Skill로 RAG 한 번, 오류를 읽고 다시 묻기 | 2 | 약 3 + 약 3~7 |
| 선택 측정 두 개 (`RUN_OPTIONAL`) | 2 | 약 80 / 약 20 |

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 왜 Skill인가: 프롬프트, 도구, Skill | 5 |
| 2 | 다섯 줄 Skill을 손으로 쓰기 | 5 |
| 3 | `SKILL.md` 규칙과 점진적 공개 | 6 |
| 4 | 설명을 바꿔 활성화 비교, 경쟁 Skill 셋 | 12 |
| 5 | 출력 형식 더하기, 참조 파일 나누기 | 7 |
| 6 | 트리거 테스트와 near-miss | 5 |
| 7 | 읽으면 안 되는 질문 | 6 |
| 8 | 출력 평가, Skill이 맡지 않는 것 | 2 |
| 9 | 직접 해보기: 내 업무 Skill | 9 |
| 10 | 정리와 확인 질문 | 3 |

셀 첫 줄의 표지는 `# [따라하기]`(그대로 실행), `# [바꿔보기]`(값을 바꾸고 예측과 비교), `# [깨뜨리기]`(일부러 실패시켜 오류 읽기), `# [내 것 만들기]`(내 값으로 교체)입니다. 이 노트북은 손으로 만든 메시지를 쓰지 않습니다.

## 1절. 왜 Skill인가

Agent에게 "회의록은 이렇게 요약하고, 표는 저렇게 계산하라"는 규칙을 모두 가르친다고 해 봅시다. 가장 쉬운 방법은 전부 **시스템 프롬프트**에 쓰는 것입니다. 그러면 질문이 한 줄짜리 사실 확인이어도 모든 규칙이 호출마다 실려 가고, 규칙끼리 부딪힙니다.

Skill은 이것을 **필요할 때만 읽는 절차**로 풉니다. 모델은 평소에 이름과 설명만 보고, 질문이 설명과 맞을 때만 본문을 읽습니다.

| | 시스템 프롬프트 | 도구 | Skill |
|---|---|---|---|
| 무엇인가 | 항상 읽히는 지침 | 부르면 코드가 실행되는 함수 | 필요할 때 읽는 절차 문서 |
| 모델에 들어가는 때 | 모든 호출에 전부 | 이름과 인자 설명은 항상 | 이름과 설명은 항상, 본문은 읽을 때 |
| 강제력 | 없음 | 실행은 코드, 부를지는 모델 | 없음. 읽을지도 모델이 정함 |
| 안 따르면 | 조용히 무시 | 오류가 돌아옴 | 오류 없음 |

비유하면 시스템 프롬프트는 벽에 붙인 규칙표, 도구는 공구함, Skill은 서랍 속 업무 매뉴얼입니다. 매뉴얼은 표지(이름과 설명)만 꽂아 두고 필요할 때 꺼내 읽습니다. 표지가 부실하면 아무도 꺼내 보지 않습니다.

마지막 줄이 이 노트북의 문제의식입니다. **Skill은 지침이지 강제가 아닙니다.** "만들었다"와 "읽혔다"는 다른 일이어서, 읽었는지를 코드로 확인합니다.

## 2절. 다섯 줄 Skill을 손으로 쓰기

먼저 설정 셀을 실행합니다. 수업에서 만드는 Skill은 모두 `outputs/skill_sandbox/` 아래에 쓰고, 제공 `skills/`는 건드리지 않습니다. 다시 실행해도 같은 상태가 되도록 sandbox를 비웁니다.

In [ ]:
import json
import shutil
from datetime import date

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from deepagents.middleware.filesystem import FilesystemPermission
from deepagents.middleware.skills import _list_skills
from langchain_core.callbacks import BaseCallbackHandler

from rag_lab.agent import TrackedSkillsBackend, ask, check_skill_folder
from rag_lab.answer import skill_spec
from rag_lab.corpus import Scope, load_corpus
from rag_lab.retrieval import Retriever
from rag_lab.settings import ROOT, announce, key_status, make_model, require_key

# 큰 측정(질문 20개 x 3회 등)은 True로 바꿔야만 돈다. False는 "이번에는 돌리지 않는다"는 선택이다.
RUN_OPTIONAL = False

SKILLS = ROOT / "skills"                      # 제공 Skill. 여기에는 쓰지 않는다.
SANDBOX = ROOT / "outputs" / "skill_sandbox"  # 수업에서 만드는 Skill은 모두 여기에 쓴다.
if SANDBOX.exists():
    shutil.rmtree(SANDBOX)
SANDBOX.mkdir(parents=True)

require_key()                                 # 키가 없으면 안내와 함께 여기서 멈춘다.
print(key_status())

Skill은 **폴더 하나와 그 안의 `SKILL.md` 파일 하나**입니다. 위 `---` 사이가 메타데이터(이름, 설명), 아래가 본문입니다. 아래 셀을 그대로 실행해 보세요. 모델 호출은 0회입니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회 — 다섯 줄 Skill 쓰기
folder = SANDBOX / "first" / "meeting-notes"
folder.mkdir(parents=True)
(folder / "SKILL.md").write_text("""---
name: meeting-notes
description: 회의록을 결정, 할 일, 미정 세 줄로 요약한다. 회의록 요약을 요청받을 때 쓴다.
---
결정, 할 일, 미정 사항을 각각 한 줄로 쓴다.
""", encoding="utf-8")
print((folder / "SKILL.md").read_text(encoding="utf-8"))

이것이 전부입니다. 코드도, 등록 절차도 없습니다. 위 네 줄이 메타데이터, 마지막 한 줄이 본문입니다. 이제 이 폴더를 Agent에게 줍니다. 아래 `skill_turn`은 (1) 이 폴더를 Skill 위치(`skills=["/"]`)로 지정한 Agent를 만들고 (2) 질문 하나를 보내고 (3) 답, 모델이 부른 도구, **`SKILL.md`를 끝까지 읽었는지**(활성화)를 돌려줍니다. 파일 쓰기는 막아 두었습니다.

In [ ]:
class FirstPrompt(BaseCallbackHandler):
    """모델이 처음 받은 시스템 메시지만 저장한다. 읽기만 하고 바꾸지 않는다."""
    def __init__(self):
        self.system = ""

    def on_chat_model_start(self, serialized, messages, **kwargs):
        if not self.system:
            content = messages[0][0].content
            self.system = content if isinstance(content, str) else "".join(b.get("text", "") for b in content)


def skill_turn(root, question):
    """root 폴더를 Skill 위치로 쓰는 Agent에 질문 하나를 보낸다."""
    seen, prompt = [], FirstPrompt()
    agent = create_deep_agent(
        model=make_model(), backend=TrackedSkillsBackend(root, seen.append), skills=["/"],
        permissions=[FilesystemPermission(operations=["write"], paths=["/**"], mode="deny")])
    result = agent.invoke({"messages": [{"role": "user", "content": question}]},
                          config={"callbacks": [prompt], "recursion_limit": 30})
    calls = [(c["name"], c["args"]) for m in result["messages"] for c in getattr(m, "tool_calls", [])]
    return {"answer": result["messages"][-1].text, "calls": calls, "seen": seen, "prompt": prompt.system}


def read_paths(turn):
    """모델이 read_file로 연 경로 목록."""
    return [args.get("file_path") for name, args in turn["calls"] if name == "read_file"]


MEETING = ("4월 7일 주간 회의. 신규 고객 온보딩 일정은 5월 12일로 확정했다. "
           "김서연이 4월 11일까지 체크리스트 초안을 만든다. 가격 개편안은 법무 검토 결과를 보고 다시 논의하기로 했다.")
print("준비 완료. 회의록 예시:", MEETING)

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 내가 쓴 Skill을 Agent가 읽는가
announce("다섯 줄 Skill을 Agent에게 준다", model_calls=3)
turn = skill_turn(SANDBOX / "first", "다음 회의록을 요약해 주세요.\n" + MEETING)
print("읽은 경로:", read_paths(turn))
print("활성화(끝까지 읽은 Skill):", turn["seen"])
print("이름이 시스템 프롬프트에 있는가:", "meeting-notes" in turn["prompt"])
print("본문 한 줄이 시스템 프롬프트에 있는가:", "각각 한 줄로 쓴다" in turn["prompt"])
print()
print(turn["answer"])

관찰할 것: `read_file`로 `/meeting-notes/SKILL.md`를 열었는지, 답이 세 줄인지, 그리고 **이름은 처음부터 시스템 프롬프트에 있지만 본문은 없다**는 점입니다. 이것이 점진적 공개의 첫 두 단계입니다. 이 한 번은 관찰이며 비율이 아닙니다. 같은 셀을 다시 실행했을 때 읽은 경로나 답의 말투가 조금 달라질 수 있다는 점도 기억해 두세요. 모델은 매번 같은 길로 가지 않습니다.

## 3절. `SKILL.md` 규칙과 점진적 공개

**규칙.** `name`은 소문자·숫자·하이픈 64자 이하이고 폴더 이름과 같아야 합니다. `description`은 1~1024자입니다. 설명은 "무엇을 + 언제 쓰는가 + 이웃과의 경계"를 담습니다. 모델이 설명만 보고 읽을지 정하기 때문입니다.

**점진적 공개 세 단계.**

1. 메타데이터(이름, 설명): 항상 시스템 프롬프트에 있습니다.
2. 본문(`SKILL.md`): 모델이 읽기로 정했을 때만 들어옵니다.
3. 참조 파일(`references/` 등): 본문이 가리킬 때만 읽습니다.

**활성화의 정의.** 이 수업에서는 `SKILL.md`를 **끝까지** 읽은 것입니다. 앞부분만 읽은 것은 읽은 것이 아닙니다. 위 `skill_turn`의 `seen`이 이 정의를 씁니다.

예를 들어 규칙을 지킨 메타데이터는 이렇게 생겼습니다.

```
---
name: meeting-notes          # 폴더 이름과 같다
description: 회의록을 ...    # 무엇을 + 언제 + (필요하면) 쓰지 않는 때
---
```

비용 면에서 보면, 메타데이터는 Skill이 열 개든 백 개든 매 호출에 실립니다. 그래서 설명은 짧고 정확해야 하고, 길고 자주 안 쓰는 설명은 본문이나 참조 파일로 보냅니다.

규칙을 어기면 어떻게 될까요? 아래 [깨뜨리기]에서 폴더 이름과 다른 `name`, 대문자·밑줄 이름, 빈 `description`을 일부러 만듭니다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회 — 규칙을 어긴 Skill 셋
BROKEN = SANDBOX / "broken"
cases = {
    "name-mismatch": ("other-name", "이름이 폴더와 다른 Skill."),
    "Bad_Name": ("Bad_Name", "대문자와 밑줄을 쓴 이름."),
    "no-description": ("no-description", ""),
}
for folder_name, (name, description) in cases.items():
    path = BROKEN / folder_name
    path.mkdir(parents=True)
    (path / "SKILL.md").write_text(
        f"---\nname: {name}\ndescription: {json.dumps(description, ensure_ascii=False)}\n---\n본문 한 줄.\n",
        encoding="utf-8")

backend = FilesystemBackend(root_dir=BROKEN, virtual_mode=True)
print("Deep Agents가 올린 목록:", [s["name"] for s in _list_skills(backend, "/")])
for folder in sorted(BROKEN.iterdir()):
    print(folder.name, "->", check_skill_folder(folder))

읽을 것: 위 셀의 영문 경고는 설치본이 내는 것입니다. 검사기(`check_skill_folder`)는 셋 모두 문제로 잡습니다. 그런데 설치본이 올린 목록에는 이름 규칙을 어긴 Skill도 들어 있을 수 있습니다. 설치본은 경고만 하고 올리기 때문입니다. **조용한 실패**이므로 Skill을 쓰면 검사기를 먼저 돌립니다.

## 4절. 설명을 바꿔 활성화 비교

모델은 설명만 보고 읽을지 정합니다. 설명을 `회의록` 한 단어로 줄이면 어떻게 될까요? 질문 두 개를 비교합니다: 직접 요청("요약해 주세요")과 돌려 말한 요청("오늘 미팅 내용 정리해 줘"). 질문당 한 번 돌리므로 **비율이 아니라 관찰**입니다.

[바꿔보기] 셀은 늘 같은 순서입니다. 바꿀 값을 확인하고, **먼저 예측을 적은 다음** 실행하고, 결과를 예측과 비교합니다. 예측이 틀렸을 때 가장 많이 배웁니다. 예측을 적지 않고 실행하면 결과가 "원래 그럴 줄 알았다"로 보이기 쉽습니다.

In [ ]:
def write_skill(root, name, description, body, files=None):
    """root/name/SKILL.md를 쓴다. files는 {상대경로: 내용}. 설명은 JSON 문자열로 감싸 YAML에서 안전하게 한다."""
    folder = root / name
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "SKILL.md").write_text(
        f"---\nname: {name}\ndescription: {json.dumps(description, ensure_ascii=False)}\n---\n{body}\n", encoding="utf-8")
    for relative, content in (files or {}).items():
        (folder / relative).parent.mkdir(parents=True, exist_ok=True)
        (folder / relative).write_text(content, encoding="utf-8")
    return folder


def activated(turn, name="meeting-notes"):
    return name in turn["seen"]


BODY = "결정, 할 일, 미정 사항을 각각 한 줄로 쓴다."
QUESTIONS = {"직접": "다음 회의록을 요약해 주세요.\n" + MEETING, "돌려 말함": "오늘 미팅 내용 정리해 줘.\n" + MEETING}

In [ ]:
# [따라하기] 유료, 모델 호출 약 6회 — 설명이 한 단어일 때
announce("설명 '회의록'으로 질문 두 개", model_calls=6)
write_skill(SANDBOX / "desc-weak", "meeting-notes", "회의록", BODY)
weak = {label: skill_turn(SANDBOX / "desc-weak", question) for label, question in QUESTIONS.items()}
for label, turn in weak.items():
    print(f"{label:6s} 활성화={activated(turn)} 읽은 경로={read_paths(turn)}")

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 6회 — description을 "무엇을 + 언제"로
# 바꿀 값: STRONG_DESCRIPTION. 먼저 예측을 적고 실행하세요: 어느 질문의 활성화가 바뀔까?
PREDICTION = "여기에 예측을 적습니다"
STRONG_DESCRIPTION = ("회의록이나 미팅 내용을 결정, 할 일, 미정 세 줄로 요약한다. "
                      "회의록 요약이나 미팅 정리를 요청받을 때 쓴다.")
announce("설명을 바꾸고 같은 질문 두 개", model_calls=6)
write_skill(SANDBOX / "desc-strong", "meeting-notes", STRONG_DESCRIPTION, BODY)
strong = {label: skill_turn(SANDBOX / "desc-strong", question) for label, question in QUESTIONS.items()}
print("예측:", PREDICTION)
print(f"{'질문':8s} {'설명=회의록':>12s} {'설명=무엇을+언제':>16s}")
for label in QUESTIONS:
    print(f"{label:8s} {str(activated(weak[label])):>12s} {str(activated(strong[label])):>16s}")

설명은 Skill의 얼굴입니다. 본문이 아무리 좋아도 설명이 부실하면 꺼내 읽히지 않습니다.

읽을 것: 표에서 두 열이 다른 질문이 있는가. 같다면 모델이 한 단어 설명으로도 충분히 짐작한 것입니다. 이 표는 질문당 한 번의 관찰이어서 "설명이 길수록 항상 활성화된다"는 결론은 낼 수 없습니다. 비율은 같은 질문을 여러 번 돌려야 말할 수 있습니다. 위 표에서 차이가 안 보였다면 이유가 하나 더 있습니다. Skill이 **하나뿐**이면 모델은 설명이 부실해도 그것밖에 고를 게 없습니다. 설명의 힘은 **경쟁자가 있을 때** 드러납니다. 아래에서 겹치는 말을 쓰는 Skill 셋을 한 폴더에 두고 비교합니다.

### 경쟁하는 Skill 셋

세 Skill(`meeting-notes` 요약, `meeting-schedule` 일정, `meeting-followup` 안내 메일)을 한 폴더에 둡니다. 설명은 두 가지 버전입니다. **막연한 설명**은 셋 모두 `회의`만 말하고, **구체적인 설명**은 무엇을 + 언제 + 쓰지 않는 때를 씁니다. 질문 네 개는 경계에 놓았습니다. 회의록이 붙은 메일 요청이나, 회의 일정 변경 안내처럼 두 Skill 사이에 걸칩니다. 질문마다 어느 Skill이 맞는지는 **출제자가 정한 기대**이고 정답표가 아닙니다. 각 조건을 세 번 돌려 질문별 활성화 횟수를 표로 봅니다. 3회는 비율이라 부르기엔 적은 수입니다. 기대와 다른 읽기가 나오면 틀렸다고 단정하기보다 질문이 정말 한쪽만 가리키는지부터 의심하세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 70회 — 막연한 설명과 구체적인 설명, 경쟁하는 Skill 셋
# 바꿀 값: PREDICTION, 그리고 아래 DESCRIPTIONS. 먼저 예측을 적고 실행하세요: 어느 질문에서 두 설명의 차이가 클까?
from concurrent.futures import ThreadPoolExecutor

PREDICTION = "여기에 예측을 적습니다"
REPEATS = 3
BODIES = {"meeting-notes": "결정, 할 일, 미정 사항을 각각 한 줄로 쓴다.",
          "meeting-schedule": "가능한 시간 후보 두 개를 제시하고 참석자 확인을 요청한다.",
          "meeting-followup": "인사, 결정, 할 일, 기한 순서로 참석자에게 보낼 안내 메일을 쓴다."}
DESCRIPTIONS = {
    "막연": {"meeting-notes": "회의 내용을 다룬다.", "meeting-schedule": "회의 일정을 다룬다.",
           "meeting-followup": "회의 후속 작업을 다룬다."},
    "구체": {"meeting-notes": "회의록 본문을 결정, 할 일, 미정 세 줄로 요약한다. 회의록이 주어지고 요약이나 정리를 요청받을 때 쓴다. 일정 조율이나 안내 메일 작성에는 쓰지 않는다.",
           "meeting-schedule": "회의 시간 후보를 정하고 참석자 일정을 맞춘다. 새 회의 시간을 잡거나 옮길 때 쓴다. 회의록 요약이나 메일 작성에는 쓰지 않는다.",
           "meeting-followup": "회의가 끝난 뒤 참석자에게 보낼 안내 메일을 쓴다. 결정과 할 일을 메일 형태로 알릴 때 쓴다. 회의록 요약만 원하거나 일정만 잡을 때는 쓰지 않는다."}}
# (질문, 출제자가 기대한 Skill)
COMPETE_QUESTIONS = [
    ("다음 회의록을 세 줄로 요약해 주세요.\n" + MEETING, "meeting-notes"),
    ("다음 주 화요일에 회의를 잡고 싶어요. 가능한 시간을 정해 주세요.", "meeting-schedule"),
    ("아래 회의 결과를 참석자에게 보낼 메일로 써 주세요.\n" + MEETING, "meeting-followup"),
    ("4월 7일 회의를 4월 9일로 옮기기로 했어요. 참석자에게 알릴 메일을 써 주세요.", "meeting-followup"),
]
for version, descriptions in DESCRIPTIONS.items():
    for name, description in descriptions.items():
        write_skill(SANDBOX / f"compete-{version}", name, description, BODIES[name])

announce("경쟁 Skill 셋 비교 (질문 4 x 설명 2 x 3회)", model_calls=70)
jobs = [(v, q, e, r) for v in DESCRIPTIONS for q, e in COMPETE_QUESTIONS for r in range(REPEATS)]
with ThreadPoolExecutor(max_workers=4) as pool:
    turns = list(pool.map(lambda j: skill_turn(SANDBOX / f"compete-{j[0]}", j[1]), jobs))
counts = {}  # (설명 버전, 질문 번호) -> {Skill: 활성화 횟수, "정확": 기대한 Skill만 읽은 횟수}
for (version, question, expected, _), turn in zip(jobs, turns):
    row = counts.setdefault((version, question), {name: 0 for name in BODIES} | {"정확": 0})
    for name in BODIES:
        row[name] += activated(turn, name)
    row["정확"] += set(turn["seen"]) == {expected}

print("예측:", PREDICTION)
print(f"{'질문':4s} {'기대':16s} {'설명':4s} " + " ".join(f"{n.split('-')[1]:>8s}" for n in BODIES) + "   기대만 읽음")
for number, (question, expected) in enumerate(COMPETE_QUESTIONS, 1):
    for version in DESCRIPTIONS:
        row = counts[(version, question)]
        print(f"Q{number:<3d} {expected:16s} {version:4s} " + " ".join(f"{row[n]}/{REPEATS:>6d}" for n in BODIES) + f"   {row['정확']}/{REPEATS}")
print("합계(기대만 읽은 횟수):", {v: sum(counts[(v, q)]["정확"] for q, _ in COMPETE_QUESTIONS) for v in DESCRIPTIONS},
      "/", REPEATS * len(COMPETE_QUESTIONS), "회 중")


읽을 것: 질문마다 `막연`과 `구체` 줄을 위아래로 비교하세요. 마지막 열이 다르면 설명이 경쟁에서 Skill을 고르는 힘을 바꾼 것입니다. 거의 같다면 이렇게 읽습니다. 이 모델은 막연한 설명에서도 Skill 이름(`meeting-schedule` 등)으로 충분히 짐작했을 수 있고, 이 질문 네 개와 3회로는 차이가 있다고도 없다고도 말하지 못합니다. **차이가 안 보인 것은 설명이 무의미하다는 증거가 아닙니다.** 질문을 더 아슬아슬하게 바꾸거나 Skill을 더 늘리면 달라질 수 있고, 그것은 `COMPETE_QUESTIONS`를 늘려 이 셀을 다시 돌리는 일입니다. 결과를 본 뒤 질문을 바꿔 다시 돌리면 차이가 나오는 질문만 골라 남기는 셈이니, 바꾸면 바꿨다고 기록하세요.

## 5절. 출력 형식 더하기, 참조 파일 나누기

Skill에 담기 좋은 것은 세 가지입니다. **절차**(이 순서로 하라), **출력 형식**(이런 모양으로 답하라), **경계**(이런 때는 쓰지 마라). 이 절에서는 출력 형식부터 더해 보고, 형식 설명이 길어지면 참조 파일로 나눕니다. 모든 셀은 같은 `meeting-notes` Skill을 조금씩 바꾼 복사본을 `outputs/skill_sandbox/` 아래 따로 둡니다.

Skill 본문에는 절차뿐 아니라 **출력 형식**도 쓸 수 있습니다. 본문에 절 제목을 지시하면 Python을 고치지 않고 답의 모양이 바뀝니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 본문에 출력 형식 더하기
announce("절 제목을 지시한 Skill", model_calls=3)
FORMAT_BODY = BODY + "\n답은 `## 결정`, `## 할 일`, `## 미정` 세 절 제목으로 나눠 쓴다."
write_skill(SANDBOX / "format", "meeting-notes", STRONG_DESCRIPTION, FORMAT_BODY)
turn = skill_turn(SANDBOX / "format", QUESTIONS["직접"])
print("활성화:", activated(turn))
print(turn["answer"])

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 3회 — 형식 지시를 표로
# 바꿀 값: TABLE_RULE. 예측: 도구 호출 수와 답 중 무엇이 바뀔까?
TABLE_RULE = "답은 `| 구분 | 내용 |` 두 열 표로 쓴다. 행은 결정, 할 일, 미정이다."
announce("형식 지시를 표로 바꾼다", model_calls=3)
write_skill(SANDBOX / "table", "meeting-notes", STRONG_DESCRIPTION, BODY + "\n" + TABLE_RULE)
table_turn = skill_turn(SANDBOX / "table", QUESTIONS["직접"])
print("도구 호출 수: 절 제목", len(turn["calls"]), "/ 표", len(table_turn["calls"]))
print("읽은 경로:", read_paths(table_turn))
print(table_turn["answer"])

절 제목 기준을 코드에서도 쓰려면 파일로 둡니다. Skill 폴더의 `output.json`에 절 제목을 쓰면 `skill_spec`이 그것을 읽습니다. 05번 노트북의 답변 검증(`validate_answer`)이 같은 파일로 절 제목을 확인합니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회 — 절 제목 기준을 output.json으로
spec_root = SANDBOX / "spec"
write_skill(spec_root, "meeting-notes", STRONG_DESCRIPTION, FORMAT_BODY)
(spec_root / "meeting-notes" / "output.json").write_text(json.dumps(
    {"sections": [{"heading": "결정", "style": "paragraph"}, {"heading": "할 일", "style": "steps"},
                  {"heading": "미정", "style": "paragraph"}]}, ensure_ascii=False, indent=2), encoding="utf-8")
spec = skill_spec("meeting-notes", spec_root)
print([(s["heading"], s["style"]) for s in spec["sections"]])

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회 — style을 허용되지 않은 값으로
bad = json.loads((spec_root / "meeting-notes" / "output.json").read_text(encoding="utf-8"))
bad["sections"][0]["style"] = "chart"
bad_root = SANDBOX / "spec-bad"
write_skill(bad_root, "meeting-notes", STRONG_DESCRIPTION, FORMAT_BODY)
(bad_root / "meeting-notes" / "output.json").write_text(json.dumps(bad, ensure_ascii=False), encoding="utf-8")
try:
    skill_spec("meeting-notes", bad_root)
except ValueError as error:
    print("ValueError:", error)

읽을 것: 허용되는 `style`은 `table`, `steps`, `paragraph`뿐입니다. 형식 기준이 파일에 있으니 잘못된 값은 모델이 아니라 코드가 먼저 잡습니다.

이제 긴 형식 설명을 **참조 파일**로 나눕니다. 본문은 짧게 두고 "형식은 `references/format.md`를 읽는다"고만 씁니다. 이것이 점진적 공개의 세 번째 단계입니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 형식 설명을 references/format.md로 옮기기
announce("참조 파일로 나눈 Skill", model_calls=3)
write_skill(SANDBOX / "split", "meeting-notes", STRONG_DESCRIPTION,
            BODY + "\n출력 형식은 /meeting-notes/references/format.md를 읽고 그대로 따른다.",
            files={"references/format.md": "답은 `## 결정`, `## 할 일`, `## 미정` 세 절 제목으로 나눠 쓴다.\n"})
split_turn = skill_turn(SANDBOX / "split", QUESTIONS["직접"])
print("읽은 경로(순서대로):", read_paths(split_turn))
print("활성화 목록:", split_turn["seen"])
print(split_turn["answer"])

관찰할 것: 읽은 경로가 `SKILL.md` 다음에 참조 파일로 이어졌는가. 활성화 목록에는 `SKILL.md`만 세어집니다.

활성화는 "끝까지 읽음"입니다. 아래 [바꿔보기]는 읽는 줄 수(`limit`)를 1000에서 3으로 줄입니다. 모델 호출은 없고 파일 읽기만 합니다.

In [ ]:
# [바꿔보기] 무료, 모델 호출 0회 — limit을 1000에서 3으로
# 바꿀 값: LIMIT. 예측: 3줄만 읽어도 활성화로 세어질까?
LIMIT = 3
for limit in (1000, LIMIT):
    seen = []
    result = TrackedSkillsBackend(SANDBOX / "format", seen.append).read("/meeting-notes/SKILL.md", 0, limit)
    print(f"limit={limit:4d} 활성화 기록={seen} next_offset={result.next_offset}")

읽을 것: `next_offset`이 비어 있지 않으면 파일이 더 남은 것이고, 그때는 활성화로 세지 않습니다. 일부만 읽은 것은 읽은 것이 아닙니다.

## 6절. 트리거 테스트와 near-miss

Skill이 잘 만들어졌는지는 **트리거 테스트**로 봅니다. 질문을 세 종류로 나눕니다.

- positive: 읽어야 하는 질문
- near-miss: 단어는 겹치지만 읽으면 안 되는 질문
- far: 아무 관련이 없는 질문

| 종류 | 예시 | 읽어야 하나 |
|---|---|---|
| positive | "다음 회의록을 요약해 주세요" | 읽어야 함 |
| near-miss | "다음 주 회의 일정을 잡아 주세요" | 읽으면 안 됨 |
| far | "오늘 날씨는 어때?" | 읽으면 안 됨 |

세는 것은 위양성(읽으면 안 되는데 읽음)과 위음성(읽어야 하는데 안 읽음)입니다. 한 번 돌리면 관찰이고, 여러 번 돌려 비율을 낼 때 비로소 신뢰할 수 있는 수가 됩니다.

## 7절. 읽으면 안 되는 질문

"다음 주 회의 일정을 잡아 주세요"는 `회의`라는 단어가 겹치지만 요약 요청이 아닙니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 단어는 겹치지만 읽으면 안 되는 질문
NEAR_MISS = "다음 주 회의 일정을 잡아 주세요."
announce("near-miss 질문 하나", model_calls=3)
near = skill_turn(SANDBOX / "desc-strong", NEAR_MISS)
print("활성화:", activated(near), "/ 읽은 경로:", read_paths(near))
print(near["answer"])

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 3회 — 설명 끝에 경계를 더하기
# 바꿀 값: BOUNDARY. 예측: 활성화가 바뀔까? 이미 안 읽었다면 같은 결과가 나옵니다.
BOUNDARY = " 회의 일정을 잡을 때는 쓰지 않는다."
announce("경계를 더한 설명", model_calls=3)
write_skill(SANDBOX / "boundary", "meeting-notes", STRONG_DESCRIPTION + BOUNDARY, BODY)
near_after = skill_turn(SANDBOX / "boundary", NEAR_MISS)
print("전:", activated(near), "/ 후:", activated(near_after))
print("읽은 경로:", read_paths(near_after))

In [ ]:
# [따라하기] 유료, 모델 호출 약 12회 — 가장 작은 트리거 테스트 (질문 4개, 각 1회)
CHECKS = [
    ("다음 회의록을 요약해 주세요.\n" + MEETING, True),
    ("오늘 미팅 내용을 결정과 할 일로 정리해 줘.\n" + MEETING, True),
    ("다음 주 회의 일정을 잡아 주세요.", False),
    ("회의실 예약은 어떻게 하나요?", False),
]
announce("트리거 테스트 4문항", model_calls=12)
false_positive, false_negative = [], []
for question, should in CHECKS:
    got = activated(skill_turn(SANDBOX / "boundary", question))
    verdict = "통과" if got == should else "실패"
    print(f"{verdict} 읽어야 함={should} 읽음={got} :: {question[:24]}")
    if got and not should:
        false_positive.append(question)
    if should and not got:
        false_negative.append(question)
print("위양성:", len(false_positive), "/ 위음성:", len(false_negative))

읽을 것: 질문이 네 개, 각 한 번이라 이것은 **트리거 테스트의 가장 작은 형태**입니다. 통과 4개가 나와도 "Skill이 안정적이다"가 아니라 "이 네 번은 맞았다"입니다. 전체 측정은 아래 선택 셀입니다.

이제 제공 Skill로 진짜 RAG 질문을 하나 던져 봅니다. 검색기와 범위는 이 셀에서 만듭니다. 범위(`Scope`), 근거 ID, 인용 검증은 04와 05에서 자세히 배웁니다. 여기서는 Skill을 읽었는가(`loaded_skills`)와 `status`만 먼저 보고, 오류가 나면 아래 설명을 따라 읽습니다. 05번 노트북의 활성화 기록으로 이어지는 연결입니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 제공 Skill(skills/)로 SLA 질문 하나
announce("제공 Skill로 RAG 한 번", model_calls=3)
retriever = Retriever(load_corpus())
scope = Scope("알파", date(2026, 9, 8))
record = ask("알파 SLA 표의 서비스 크레딧 비율 중 최댓값은 무엇인가요?",
             retriever=retriever, scope=scope, model=make_model(), max_attempts=1)
print("상태:", record["status"], "/ 검증:", record["validation"], "/ 오류:", record["errors"])
print("활성화(loaded_skills):", record["loaded_skills"])
for section in (record["answer"] or {}).get("sections", []):
    for item in section["items"]:
        print("답 문장:", item["text"], "/ 연결한 근거:", item["evidence_ids"])
print("skill_read 이벤트:", [e for e in record["trace"]["events"] if e["event"] == "skill_read"])

위 셀에서 `status`가 `error`여도 놀라지 마세요. `인용문에 없는 숫자: …`가 보이면 먼저 **어느 문장의 어떤 숫자인지** 찾으세요. 검증은 답 문장의 숫자가 그 문장에 연결한 인용문에 있는지 봅니다. 저자가 이 셀의 질문을 9번 돌렸을 때 7번 통과하고 2번 거부됐습니다(`인용문에 없는 숫자: 20%`). 표를 그대로 옮긴 문장은 `20%`가 있는 근거에 연결돼 있어 통과했습니다. 거부된 문장은 `추가 약정 A-01은 신청 기한만 바꾸므로 최댓값은 여전히 20%입니다`처럼 말하면서 **약정 근거에만** 연결한 문장이었습니다. 약정 문서에는 `20%`가 없습니다. 내용이 맞아도 그 숫자가 있는 SLA 근거를 같은 문장에 연결하지 않으면 코드가 확인할 수 없으므로 맞게 거부한 것입니다. 표를 읽고 계산해 만든 숫자도 같은 이유로 거부됩니다. 모델이 매번 같은 답을 내지는 않으므로, 이 셀의 결과가 다르면 그 숫자가 어느 문장, 어느 근거에 연결됐는지 직접 확인하세요. 활성화 기록(`loaded_skills`)은 Skill을 읽었다는 증거이고, `status`와 `validation`은 답이 인용 검증을 통과했는지 따로 말해 줍니다. 읽었다고 답이 맞는 것은 아닙니다. 검증 실패의 이유는 `errors`에 그대로 나옵니다.

## 8절. 출력 평가, Skill이 맡지 않는 것

트리거 테스트는 "읽었는가"를 봅니다. **출력 평가**는 "읽은 덕에 답이 나아졌는가"를 봅니다. 같은 질문을 Skill이 있을 때와 없을 때 돌려 답의 형식, 인용 수, 호출 수, 토큰을 비교합니다. 선택 셀에 있습니다.

Skill이 **맡지 않는 것**도 알아 둡니다. 호출 한도(예산)는 코드가 막고, 범위(고객·기준일)는 검색 도구가 걸러 주고, 근거가 정말 원문에 있는지는 검증 코드가 확인합니다. Skill에 "예산을 넘기지 마라"고 써도 강제되지 않습니다.

그래서 역할을 나누면 이렇습니다. Skill은 **어떻게 일할지**(순서, 형식, 경계)를 알려 주고, 코드는 **넘으면 안 되는 선**(예산, 범위, 근거 검증)을 지킵니다. 05번 노트북이 이 둘을 한 Agent에 묶습니다. 이 노트북에서 만든 Skill은 가볍고 약해서 모델이 안 읽어도 Agent는 아무 오류 없이 답합니다. 그 사실을 눈으로 확인한 것이 오늘의 가장 큰 수확입니다.

오류의 원인을 확인했으면 어떻게 할까요? 질문을 쉽게 바꾸는 것은 답이 아닙니다. 숫자는 원문에 있고, 문제는 문장과 근거의 연결이기 때문입니다. 아래 셀은 **같은 질문**을 `ask`의 기본값 `max_attempts=2`로 돌립니다. 첫 답이 검증에 실패하면 오류를 모델에게 돌려주고 한 번 더 시도합니다. 검증기와 예산은 그대로이고, 재시도의 호출도 같은 예산 안에서 셉니다.

재시도는 첫 답이 실패할 때만 일어납니다. 그래서 이 셀에서 재시도가 0번이면 재시도의 효과가 아니라 이번 첫 답이 통과한 것뿐입니다. 저자 실행에서 재시도를 켠 13번(질문 두 가지)은 모두 통과했고, 그중 5번은 첫 답이 거부된 뒤 재시도로 통과했습니다. 표본이 작아 보장은 아닙니다. 둘째 시도까지 실패하면 `error`로 끝나고, 어느 문장이 문제인지 출력합니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 3~7회 — 질문은 그대로 두고, 검증 오류를 돌려주는 재시도를 켠다
# 바꿀 값: MAX_ATTEMPTS. 예측: 앞 셀의 오류가 다시 나올까? 재시도는 몇 번 일어날까? 1로 되돌리면 앞 셀과 같은 조건이다.
MAX_ATTEMPTS = 2
announce("같은 질문, 오류를 돌려주고 한 번 더", model_calls=5)
fixed = ask(record["question"], retriever=retriever, scope=scope, model=make_model(), max_attempts=MAX_ATTEMPTS)
failed_tries = [event["errors"][0] for event in fixed["trace"]["events"] if event["event"] == "validation_retry"]
print("앞 셀(재시도 없음):", record["status"], record["errors"][:1])
print("검증에 실패한 시도:", len(failed_tries), failed_tries)
print("이번 결과:", fixed["status"], "/ 검증:", fixed["validation"], "/ 오류:", fixed["errors"])
if fixed["status"] == "error":
    for section in (fixed["answer"] or {}).get("sections", []):
        for item in section["items"]:
            print("답 문장:", item["text"], "/ 연결한 근거:", item["evidence_ids"])
print(fixed["markdown"])


## 직접 해보기

내 업무에서 반복되는 요약이나 정리 하나를 Skill로 만듭니다. 아래 세 셀에는 **예시 값(주간 보고 요약)**이 채워져 있어 위에서 아래로 그대로 돕니다. 값을 내 것으로 바꾸고 다시 실행하세요. 값을 비우면 안내가 나옵니다. 순서는 (1) Skill 쓰고 검사 (2) 읽어야 하는 질문 (3) 읽으면 안 되는 질문입니다.

In [ ]:
# [내 것 만들기] 무료, 모델 호출 0회 — 내 Skill 쓰고 검사하기
MY_NAME = "weekly-report"
MY_DESCRIPTION = "한 주 업무 메모를 완료, 진행 중, 막힘 세 줄로 요약한다. 주간 보고 요약을 요청받을 때 쓴다."
MY_BODY = "완료, 진행 중, 막힘을 각각 한 줄로 쓴다. 막힘이 없으면 '없음'이라고 쓴다."

MINE = SANDBOX / "mine"
if MINE.exists():
    shutil.rmtree(MINE)
if not (MY_NAME.strip() and MY_DESCRIPTION.strip() and MY_BODY.strip()):
    print("MY_NAME(소문자·숫자·하이픈), MY_DESCRIPTION(무엇을 + 언제), MY_BODY(절차 한두 줄)를 채우세요.")
else:
    write_skill(MINE, MY_NAME.strip(), MY_DESCRIPTION.strip(), MY_BODY.strip())
    problems = check_skill_folder(MINE / MY_NAME.strip())
    print("검사 결과:", problems or "문제 없음")

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 3회 — 읽어야 하는 질문
MY_QUESTION = "이번 주 메모를 주간 보고로 요약해 줘. 신규 견적 3건 발송 완료, 보고서 양식 개편 진행 중, 승인권자 휴가로 계약서 서명 대기."
if not MY_QUESTION.strip() or not (MINE / MY_NAME.strip() / "SKILL.md").is_file():
    print("앞 셀에서 Skill을 만들고, MY_QUESTION에 읽어야 하는 질문을 쓰세요.")
else:
    announce("내 Skill에 읽어야 하는 질문", model_calls=3)
    mine = skill_turn(MINE, MY_QUESTION)
    print("활성화:", MY_NAME.strip() in mine["seen"], "/ 읽은 경로:", read_paths(mine))
    print(mine["answer"])

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 3회 — 읽으면 안 되는 질문
MY_NEAR_MISS = "다음 주 주간 회의 시간을 목요일 오후로 옮겨 주세요."
if not MY_NEAR_MISS.strip() or not (MINE / MY_NAME.strip() / "SKILL.md").is_file():
    print("MY_NEAR_MISS에 단어는 겹치지만 읽으면 안 되는 질문을 쓰세요.")
else:
    announce("내 Skill에 near-miss 질문", model_calls=3)
    mine_near = skill_turn(MINE, MY_NEAR_MISS)
    got = MY_NAME.strip() in mine_near["seen"]
    print("활성화:", got, "/ 읽은 경로:", read_paths(mine_near))
    if got:
        print("읽으면 안 되는 질문을 읽었습니다. MY_DESCRIPTION 끝에 '...할 때는 쓰지 않는다'를 더해 다시 실행하세요.")

## 선택 측정

아래 두 셀은 표지가 없고 `RUN_OPTIONAL = True`일 때만 돕니다. 한 시간 수업에서 기다릴 수 없는 규모라서 **돌리지 않기로 한 선택**입니다. `False`면 돌리지 않았다는 사실만 출력합니다.

In [ ]:
if not RUN_OPTIONAL:
    print("전체 트리거 측정(질문 20개 x 3회, 저자의 1회 실행에서 모델 호출 81회)은 이번에는 돌리지 않았습니다.")
else:
    TRIGGER_SET = [(f"다음 회의록을 요약해 주세요.\n{MEETING}", True), (f"미팅 내용 정리해 줘.\n{MEETING}", True),
                   (f"이 회의 기록에서 할 일만 뽑아 줘.\n{MEETING}", True), (f"회의록 세 줄 요약 부탁해요.\n{MEETING}", True),
                   (f"아래 회의에서 정해진 것이 뭐야?\n{MEETING}", True), (f"회의 결과를 짧게 보고서용으로 써 줘.\n{MEETING}", True),
                   ("다음 주 회의 일정을 잡아 주세요.", False), ("회의실 예약은 어떻게 하나요?", False),
                   ("회의 시간을 오후로 옮겨 주세요.", False), ("회의 참석자에게 초대장을 보내 줘.", False),
                   ("회의 자료 템플릿을 만들어 줘.", False), ("회의 중 쓸 영어 표현을 알려 줘.", False),
                   ("파이썬 리스트를 정렬하는 법은?", False), ("오늘 날씨는 어때?", False), ("점심 메뉴를 추천해 줘.", False),
                   ("2의 10제곱은?", False), ("서울에서 부산까지 거리는?", False), ("이메일 제목을 지어 줘.", False),
                   ("SQL JOIN을 설명해 줘.", False), ("한국의 수도는?", False)]
    announce("전체 트리거 측정", model_calls=80)
    for question, should in TRIGGER_SET:
        runs = [activated(skill_turn(SANDBOX / "boundary", question)) for _ in range(3)]
        rate = sum(runs) / len(runs)
        print(f"읽어야 함={should} 비율={rate:.2f} ({len(runs)}회) :: {question[:24]}")

In [ ]:
if not RUN_OPTIONAL:
    print("Skill 있음/없음 비교(질문 3개 x 두 조건, 저자의 1회 실행에서 모델 호출 18회)는 이번에는 돌리지 않았습니다.")
else:
    announce("Skill 있음과 없음 비교", model_calls=20)
    none_dir = SANDBOX / "no-skills"
    none_dir.mkdir()
    retriever = Retriever(load_corpus())
    scope = Scope("알파", date(2026, 9, 8))
    for question in ["알파 SLA 표의 서비스 크레딧 비율 중 최댓값은 무엇인가요?", "알파 SLA에서 장애 신고는 몇 시간 안에 해야 하나요?",
                     "알파 계약의 서비스 크레딧 신청 기한은 언제인가요?"]:
        for label, skills_dir in (("Skill 있음", SKILLS), ("Skill 없음", none_dir)):
            rec = ask(question, retriever=retriever, scope=scope, model=make_model(), max_attempts=1, skills_dir=skills_dir)
            counts = rec["trace"]["counts"]
            print(f"{label} 상태={rec['status']} 검증={rec['validation']} 모델호출={counts.get('model', 0)} 오류={rec['errors'][:1]} :: {question[:20]}")

## 확인 질문

1. `SKILL.md`의 메타데이터와 본문 중, 모델이 처음부터 보는 것은 무엇인가요?
2. 활성화의 정의는 무엇이고, 앞 세 줄만 읽은 경우는 왜 활성화가 아닌가요?
3. 설명을 바꿔 활성화가 달라졌다면, 질문당 한 번의 관찰로 무엇까지 말할 수 있나요?
4. near-miss 질문이 필요한 이유를 하나의 예로 설명하세요.
5. 호출 한도와 범위 필터는 왜 Skill이 아니라 코드가 맡나요?

## 정리

- Skill은 이름과 설명이 항상 보이고, 본문은 읽을 때만, 참조 파일은 본문이 가리킬 때만 읽힙니다.
- 설명은 "무엇을 + 언제 + 경계"입니다. 형식은 본문에, 큰 설명은 참조 파일에 둡니다.
- Skill은 강제가 아닙니다. 읽혔는지는 코드로 확인하고, 한 번의 활성화는 비율이 아니라 관찰입니다.
- 읽으면 안 되는 질문(near-miss)까지 시험해야 트리거 테스트입니다.
- 다음 노트북(04)에서는 검색을 설계하고 어려운 질문으로 측정합니다.